In [2]:
!pip install coremltools onnx addict jiwer datasets
import torch
from transformers import AutoModel, AutoProcessor, AutoTokenizer
import onnx
import coremltools as ct
from pathlib import Path
from abc import ABC, abstractmethod
from PIL import Image
import numpy as np
from jiwer import cer, wer
from time import time
from tqdm import tqdm
import pandas as pd

import warnings
warnings.filterwarnings(action='ignore')

In [3]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Classes for each model

In [ ]:
class AbstractModel(ABC):
    def __init__(self, model_id, output_dir):
        self.model_id = model_id
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True, parents=True)
        self.model = None
        self.processor = None

    @property
    @abstractmethod
    def model_name(self):
        """Getter for model name"""
        pass

    @abstractmethod
    def load_model(self):
        """Method to import model & set it on eval mode"""
        pass

    @abstractmethod
    def prepare_for_inference(self):
        """Loading model, setting to active device"""
        pass

    @abstractmethod
    def predict(self, image_path, prompt=None):
        """
        Run inference on single image
        Returns: (text, inference_time)
        """
        pass

    def export_to_onnx(self):
        """Setting up model to onnx format - может не работать для VLM"""
        raise NotImplementedError(
            f"ONNX export for {self.model_name} requires custom implementation"
        )

    def to_coreml(self):
        """Exporting model to .mlmodel fmt - может не работать для VLM"""
        raise NotImplementedError(
            f"CoreML export for {self.model_name} requires custom implementation"
        )

### PaddleOCR

In [ ]:
# 3 448 448
class PaddleModel(AbstractModel):
    @property
    def model_name(self):
        return "PaddleOCR-VL"

    def load_model(self):
        self.processor = AutoProcessor.from_pretrained(
            self.model_id,
            trust_remote_code=True
        )

        self.model = AutoModel.from_pretrained(
            self.model_id,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )

        self.model.eval()
        print(f"Loaded {self.model_id}")

    def prepare_for_inference(self):
        if self.model is None:
            self.load_model()

        self.model = self.model.to(DEVICE)
        print(f"Model ready for inference {self.model_id}")

    def predict(self, image_path, prompt="Extract all text from this image"):
        if self.model is None:
            self.prepare_for_inference()

        start = time()
        image = Image.open(image_path).convert('RGB')

        # PaddleOCR-VL специфичный формат
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False
            )

        text = self.processor.decode(outputs[0], skip_special_tokens=True)
        inference_time = time() - start

        return text, inference_time

### QwenV3

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor

# 3 448 448
class Qwen3VLModel(AbstractModel):
    @property
    def model_name(self):
        return "Qwen3-VL-2B"


    def load_model(self):
        self.processor = Qwen2VLProcessor.from_pretrained(
            self.model_id,
            trust_remote_code=True
        )
        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            self.model_id,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )

        self.model.eval()
        print(f"Model loaded {self.model_id}")


    def prepare_for_inference(self):
        if self.model is None:
            self.load_model()

        self.model = self.model.to("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Model ready for inference {self.model_id}")


    def predict(self, image_path, prompt="Extract all text"):
        if self.model is None:
            self.prepare_for_inference()

        start = time()
        image = Image.open(image_path).convert('RGB')

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

        text_prompt = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.processor(
            text=[text_prompt],
            images=[image],
            padding=True,
            return_tensors="pt"
        ).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False
            )

        text = self.processor.decode(outputs[0], skip_special_tokens=True)
        inference_time = time() - start

        return text, inference_time

### LightOn

In [ ]:
class LightOnModel(AbstractModel):
    @property
    def model_name(self):
        return "LightOn-OCR"

    def load_model(self):
        self.processor = AutoProcessor.from_pretrained(
            self.model_id,
            trust_remote_code=True
        )
        self.model = AutoModel.from_pretrained(
            self.model_id,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        self.model.eval()
        print(f"Loaded {self.model_id}")

    def prepare_for_inference(self):
        if self.model is None:
            self.load_model()
        self.model = self.model.to("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Model ready for inference {self.model_id}")

    def predict(self, image_path, prompt=None):
        if self.model is None:
            self.prepare_for_inference()

        start = time()

        image = Image.open(image_path).convert('RGB')
        inputs = self.processor(images=image, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=512)

        text = self.processor.decode(outputs[0], skip_special_tokens=True)
        inference_time = time() - start

        return text, inference_time

### DeepSeek AI

In [ ]:
class DeepseekModel(AbstractModel):
    @property
    def model_name(self):
        return "DeepSeek-OCR"

    def load_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_id,
            trust_remote_code=True
        )
        self.model = AutoModel.from_pretrained(
            self.model_id,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        self.model.eval()
        print(f"Loaded {self.model_id}")

    def prepare_for_inference(self):
        if self.model is None:
            self.load_model()
        self.model = self.model.to("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Model ready for inference {self.model_id}")

    def predict(self, image_path, prompt=None):
        """ai written"""
        if self.model is None:
            self.prepare_for_inference()

        start = time()

        image = Image.open(image_path).convert('RGB')
        # DeepSeek-специфичный API
        inputs = self.model.prepare_inputs(image, self.tokenizer)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=512)

        text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        inference_time = time() - start

        return text, inference_time

## Actual fabric wtf

In [ ]:
class ModelFabric:
    ALL_MODELS = {
        "paddle": PaddleModel,
        "qwen": Qwen3VLModel,
        "lighton": LightOnModel,
        "deepseek": DeepseekModel,
    }

    @staticmethod
    def setup_model(model_name, output_dir="./models"):
        selected_model = ModelFabric.ALL_MODELS[model_name](output_dir=output_dir)

        selected_model.load_model()
        return selected_model

    @staticmethod
    def list_models():
        print("Available models:")
        for k in ModelFabric.ALL_MODELS.keys():
            print(k, end = " | ")